# Task 2 — three seeds, mean +- std

The single-seed run gave Attention U-Net 0.7055 Jaccard against U-Net 0.6887. That gap
is believable but it is one run on a 5-image test set, so it cannot carry a claim on its
own. This notebook trains all three models on **seeds 27, 1 and 2** and reports
mean +- standard deviation, which can.

Everything else is unchanged from the converged run: same split, same 256 crops, same
Dice+BCE, `n_epochs = 150` with patience 5 so early stopping decides when to stop.

## Expect about 1.5 to 2 hours

U-Net++ is the slow one, roughly 17 minutes per seed. The seed loop is on the **outside**
on purpose: after each seed finishes, a running table prints. If Colab disconnects after
seed 2 you still have two complete seeds to report, instead of nothing.

## How to run

1. **Runtime -> Change runtime type -> T4 GPU -> Save**
2. **Runtime -> Run all**
3. At "Choose Files" upload **`module2_code.zip`** only (16 KB). The membrane data
   downloads itself.
4. Leave the tab open and the Mac awake.

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
      or "NO GPU — Runtime > Change runtime type > T4 GPU")

## Upload `module2_code.zip`

In [ ]:
import os, shutil, subprocess, sys, zipfile
from google.colab import files

WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
os.chdir("/content")

for name in files.upload():
    with zipfile.ZipFile(f"/content/{name}") as z:
        z.extractall(WORK)
    print("unpacked", name)

os.chdir(WORK)
sys.path.insert(0, WORK)
if not os.path.isdir(f"{WORK}/src"):
    raise FileNotFoundError("no src/ — that was not module2_code.zip")
print("code:", sorted(os.listdir("src")))

In [ ]:
if not os.path.isdir("data/train/images"):
    subprocess.run(["git", "clone", "--depth", "1", "-q",
                    "https://github.com/zhixuhao/unet.git", "/tmp/unet_repo"], check=True)
    src = "/tmp/unet_repo/data/membrane"
    for dst, pattern in [("data/train/images", "train/image"),
                         ("data/train/labels", "train/label")]:
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(f"{src}/{pattern}"):
            shutil.copy(f"{src}/{pattern}/{f}", dst)
    os.makedirs("data/test", exist_ok=True)
    for f in os.listdir(f"{src}/test"):
        if not f.endswith("_predict.png"):
            shutil.copy(f"{src}/test/{f}", "data/test")

print("membrane:", len(os.listdir("data/train/images")), "images")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from src.attention_unet import AttentionUNet, UNet
from src.losses import LOSSES
from src.membrane import load_membrane
from src.results import save_run
from src.seed import get_device, set_seed
from src.unetpp import UNetPlusPlus
import src.segmentation as seg

device = get_device()
print("device:", device)

MODELS = {
    "Attention U-Net": lambda: AttentionUNet(base=64),
    "U-Net":           lambda: UNet(base=64),
    "U-Net++":         lambda: UNetPlusPlus(base=64),
}
SEEDS = [27, 1, 2]
CFG = dict(crop_size=256, batch_size=4, lr=1e-4, n_epochs=150, patience=5,
           repeats=8, loss="dice_bce", splits=(20, 5, 5))
METRICS = ["jaccard", "f1", "recall", "precision", "accuracy"]


def summary_table(collected):
    """mean +- std across whatever seeds have finished so far."""
    n = len(next(iter(collected.values())))
    lines = [f"Seeds completed: {n}",
             "| Model | " + " | ".join(METRICS) + " |",
             "|---" * (len(METRICS) + 1) + "|"]
    for model, runs in collected.items():
        cells = []
        for k in METRICS:
            values = [r[k] for r in runs]
            cells.append(f"{np.mean(values):.4f} ± {np.std(values):.4f}"
                         if n > 1 else f"{values[0]:.4f}")
        lines.append(f"| {model} | " + " | ".join(cells) + " |")
    return "\n".join(lines)

## Train

Outer loop is the seed. After every seed the running table prints, so a disconnect
costs you only the seed in progress.

In [ ]:
collected = {name: [] for name in MODELS}       # held-out 5 images
collected_train = {name: [] for name in MODELS} # training set (what the exercise asks for)
stop_info = []

for seed in SEEDS:
    for name, factory in MODELS.items():
        set_seed(seed)
        print(f"\n{'='*70}\n{name}   seed {seed}\n{'='*70}")
        tr, va, te, tr_eval = load_membrane(root="data", crop_size=CFG["crop_size"],
                                            batch_size=CFG["batch_size"], splits=CFG["splits"],
                                            repeats=CFG["repeats"], seed=seed,
                                            include_train_eval=True)
        model = factory().to(device)
        criterion = LOSSES[CFG["loss"]]()
        optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                               factor=0.5, patience=3)
        tag = name.replace(" ", "_").replace("+", "p")
        history = seg.fit(model, tr, va, criterion, optimizer, device,
                          n_epochs=CFG["n_epochs"], patience=CFG["patience"],
                          scheduler=scheduler, checkpoint_path=f"{WORK}/{tag}_s{seed}.pt")
        metrics, _ = seg.evaluate(model, te, criterion, device)
        train_metrics, _ = seg.evaluate(model, tr_eval, criterion, device)

        collected[name].append(metrics)
        collected_train[name].append(train_metrics)
        stop_info.append((name, seed, history["best_epoch"], len(history["train_loss"])))
        save_run(tag=f"{tag}_s{seed}", config={**CFG, "model": name, "seed": seed},
                 metrics={"held_out": metrics, "train": train_metrics},
                 history=history, model=model)
        print({k: round(v, 4) for k, v in metrics.items()})
        if name == "Attention U-Net" and seed == SEEDS[0]:
            figure_model, figure_loader = model, te

    print(f"\n\n########## after seed {seed} ##########")
    print("TRAINING SET (report this):")
    print(summary_table(collected_train))
    print("\nHELD-OUT 5 IMAGES (for the discussion):")
    print(summary_table(collected))

## Did every run converge?

In [ ]:
capped = [s for s in stop_info if s[2] >= s[3]]
for name, seed, best, ran in stop_info:
    flag = "" if best < ran else "   <-- STILL CAPPED"
    print(f"{name:18s} seed {seed}   best epoch {best:3d} of {ran:3d}{flag}")
print("\nall converged" if not capped else f"\n{len(capped)} run(s) hit the cap — raise n_epochs")

## Table 2 — paste this into the report

The updated exercise says: *"Since the Membrane dataset does not provide ground-truth
masks for the test images, please report the evaluation metrics only on the training
set."* So the **first** table is Table 2.

The second table is metrics on 5 held-out images the models never trained on. It is not
what was asked for, but it is the honest measure of generalisation, and the gap between
the two tables is worth one sentence in the discussion.

In [ ]:
print("TABLE 2 — training set")
print(summary_table(collected_train))
print()
print("Held-out 5 images, for the discussion only")
print(summary_table(collected))

### Is the difference real?

A gap is only worth claiming if it is larger than the spread across seeds. This prints
the Jaccard gap next to the pooled standard deviation.

In [ ]:
jac = {m: np.array([r["jaccard"] for r in runs]) for m, runs in collected_train.items()}
ranked = sorted(jac.items(), key=lambda kv: -kv[1].mean())
best_name, best_vals = ranked[0]

print(f"best: {best_name}  {best_vals.mean():.4f} ± {best_vals.std():.4f}\n")
for name, vals in ranked[1:]:
    gap = best_vals.mean() - vals.mean()
    pooled = np.sqrt((best_vals.std()**2 + vals.std()**2) / 2)
    verdict = "larger than the seed noise" if gap > pooled else "WITHIN the seed noise"
    print(f"vs {name:18s} gap {gap:+.4f}   pooled std {pooled:.4f}   -> {verdict}")

## Figures

In [ ]:
figure_model.eval()
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
with torch.no_grad():
    for col, (image, mask) in zip(range(3), figure_loader):
        prob = torch.sigmoid(figure_model(image.to(device)))[0, 0].cpu()
        for row, (img, title) in enumerate([(image[0, 0], "original"),
                                            (mask[0, 0], "ground truth"),
                                            (prob >= 0.5, "prediction")]):
            axes[row][col].imshow(img, cmap="gray")
            axes[row][col].set_title(title); axes[row][col].axis("off")
fig.tight_layout(); fig.savefig("task2_qualitative.png", dpi=150)

image, _ = next(iter(figure_loader))
with torch.no_grad():
    _, attention = figure_model(image.to(device), return_attention=True)
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(image[0, 0], cmap="gray"); axes[0].set_title("input")
for ax, a in zip(axes[1:], attention):
    ax.imshow(a[0, 0].cpu(), cmap="inferno"); ax.set_title(f"gate @ {tuple(a.shape[-2:])}")
for ax in axes: ax.axis("off")
fig.tight_layout(); fig.savefig("task2_attention.png", dpi=150)

## Download

In [ ]:
import datetime
out = "/content/out"
shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
shutil.copytree(f"{WORK}/results", f"{out}/results")
for png in [p for p in os.listdir(WORK) if p.endswith(".png")]:
    shutil.copy(f"{WORK}/{png}", out)
with open(f"{out}/table2.md", "w") as f:
    f.write("TABLE 2 — training set\n" + summary_table(collected_train)
            + "\n\nHeld-out 5 images\n" + summary_table(collected))
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
files.download(shutil.make_archive(f"/content/task2_seeds_{stamp}", "zip", out))

---
### If Colab disconnects partway

Copy the last printed `########## after seed N ##########` table out of the output.
Two seeds is still far better than one, and reporting "mean +- std over 2 seeds" is
honest. Do not silently report it as three.